# Deep Q-Network (DQN)
---
In this notebook, you will implement a DQN agent with OpenAI Gym's LunarLander-v2 environment.

### 1. Import the Necessary Packages

In [4]:
!pip install gymnasium[Box2D] swig

  Using cached swig-4.4.1-py3-none-win_amd64.whl.metadata (3.5 kB)
  Using cached Box2D-2.3.10-cp313-cp313-win_amd64.whl.metadata (593 bytes)
Using cached Box2D-2.3.10-cp313-cp313-win_amd64.whl (1.3 MB)
Using cached swig-4.4.1-py3-none-win_amd64.whl (2.5 MB)
   ---------------------------------------- 0.0/9.7 MB ? eta -:--:--
   --- ------------------------------------ 0.8/9.7 MB 5.9 MB/s eta 0:00:02
   ------- -------------------------------- 1.8/9.7 MB 5.5 MB/s eta 0:00:02
   ---------------- ----------------------- 3.9/9.7 MB 7.4 MB/s eta 0:00:01
   -------------------------- ------------- 6.3/9.7 MB 8.7 MB/s eta 0:00:01
   ---------------------------------- ----- 8.4/9.7 MB 9.0 MB/s eta 0:00:01
   ---------------------------------------- 9.7/9.7 MB 9.2 MB/s  0:00:01

   ---------------------------------------- 0/3 [swig]
   ---------------------------------------- 0/3 [swig]
   ---------------------------------------- 0/3 [swig]
   ---------------------------------------- 0/3 [swig

In [ ]:
import gymnasium
import random
import torch
import numpy as np
from collections import deque
import matplotlib.pyplot as plt
%matplotlib inline

### 2. Instantiate the Environment and Agent

Initialize the environment in the code cell below.

In [ ]:
# ======================== INITIALIZE LUNARLANDER ENVIRONMENT ========================
# LunarLander-v3 is a continuous control environment, but we'll use discrete actions (4)
# The agent controls a lunar lander trying to land safely on the moon's surface

env = gymnasium.make('LunarLander-v3')
env.reset(seed=0)

# ======================== STATE SPACE ========================
# The state is an 8-dimensional continuous observation vector representing:
# 1. x - Horizontal position of lander (-1.0 to 1.0, where 0 is center)
# 2. y - Vertical position of lander (-1.0 to 1.0, where 0 is ground level)
# 3. vx - Horizontal velocity (-5 to 5 m/s, changes based on thrusters)
# 4. vy - Vertical velocity (-5 to 5 m/s, negative means falling)
# 5. angle - Rotation angle of lander in radians (-π to π, 0 = upright)
# 6. angular_velocity - Rate of rotation (-8 to 8 rad/s)
# 7. left_leg_contact - Boolean (0 or 1): is left leg touching ground?
# 8. right_leg_contact - Boolean (0 or 1): is right leg touching ground?

print('State shape: ', env.observation_space.shape)  # Output: (8,) - 8-dimensional state vector

# ======================== ACTION SPACE ========================
# For this DQN implementation, we use 4 DISCRETE actions (not continuous):
# 0 - Do nothing (no engines fire)
# 1 - Fire left orientation engine (counter-clockwise rotation)
# 2 - Fire main engine downward (decelerate descent)
# 3 - Fire right orientation engine (clockwise rotation)
#
# Note: The actual environment (v3) supports continuous thrust control,
# but we discretize to 4 actions for this DQN implementation.

print('Number of actions: ', env.action_space.n)  # Output: 4 (discrete action space)

State shape:  (8,)
Number of actions:  4


Please refer to the instructions in `Deep_Q_Network.ipynb` if you would like to write your own DQN agent.  Otherwise, run the code cell below to load the solution files.

In [ ]:
# ======================== IMPORT AND INSTANTIATE AGENT ========================
from dqn_agent import Agent

# Create a DQN Agent with:
# - state_size=8: Matches LunarLander's 8-dimensional state vector
#   [x, y, vx, vy, angle, angular_velocity, left_leg_contact, right_leg_contact]
# - action_size=4: Matches our discrete action set (do nothing, left engine, main engine, right engine)
# - seed=0: Ensures reproducible random initialization for neural network weights
agent = Agent(state_size=8, action_size=4, seed=0)

# ======================== WATCH UNTRAINED AGENT ========================
# Run the agent in the environment WITHOUT training to see random behavior
state, _ = env.reset()

# Execute 200 timesteps in the environment
for j in range(200):
    # Agent selects an action using current (untrained) policy
    # With no exploration noise (default eps=0), it follows the network's greedy action
    action = agent.act(state)
    
    # Render the environment (visualize the lander)
    env.render()
    
    # Execute the action and receive:
    # - next_state: the resulting 8-dimensional state vector after action
    # - reward: numerical feedback (-1 per step, +10 for each leg landing, -0.3 for main engine)
    # - terminated: True if crashed or landed (episode ends)
    # - truncated: True if max timesteps reached
    # - _: info dict (unused in DQN)
    state, reward, terminated, truncated, _ = env.step(action)
    
    # Exit episode if terminal state reached
    if terminated or truncated:
        break

env.close()

### 3. Train the Agent with DQN

Run the code cell below to train the agent from scratch.  You are welcome to amend the supplied values of the parameters in the function, to try to see if you can get better performance!

Alternatively, you can skip to the next step below (**4. Watch a Smart Agent!**), to load the saved model weights from a pre-trained agent.

In [ ]:
def dqn(n_episodes=2000, max_t=1000, eps_start=1.0, eps_end=0.01, eps_decay=0.995):
    """Deep Q-Learning training function.
    
    This implements the complete DQN training loop with experience replay and epsilon-greedy exploration.
    The agent learns by interacting with the environment, storing experiences, and performing
    gradient updates on random batches from the replay buffer.

    Params
    ======
        n_episodes (int): maximum number of training episodes (2000 episodes)
        max_t (int): maximum number of timesteps per episode (1000 steps max)
        eps_start (float): starting value of epsilon (1.0 = 100% exploration)
        eps_end (float): minimum value of epsilon (0.01 = 1% exploration minimum)
        eps_decay (float): multiplicative factor (per episode) for decreasing epsilon (decay rate)
    """
    
    # ======================== INITIALIZE TRAINING DATA STRUCTURES ========================
    # List to store the episode return (sum of rewards) for each episode
    # Used for tracking learning progress and plotting performance over time
    scores = []
    
    # Rolling window deque to store the last 100 episode scores
    # Used to determine when environment is "solved" (avg score >= 200 for 100 episodes)
    # maxlen=100 means oldest scores are automatically removed when exceeding capacity
    scores_window = deque(maxlen=100)
    
    # Initialize epsilon for epsilon-greedy action selection
    # Starts at 1.0 (100% random exploration) and decays toward eps_end
    # Higher epsilon = more exploration (random actions)
    # Lower epsilon = more exploitation (greedy actions from Q-network)
    eps = eps_start
    
    # ======================== MAIN TRAINING LOOP: ITERATE OVER EPISODES ========================
    # Each episode is one complete interaction with the environment from start to terminal state
    for i_episode in range(1, n_episodes+1):
        # Reset environment and get initial state
        # Returns initial state observation and info dict
        state, _ = env.reset()
        
        # Initialize episode return counter - accumulates rewards during this episode
        # Will be added to scores list at end of episode
        score = 0
        
        # ======================== EPISODE LOOP: EXECUTE TIMESTEPS ========================
        # Each timestep: perceive state, select action, execute in environment, observe result
        # This continues until episode terminates (crash/land) or max_t steps reached
        for t in range(max_t):
            # STEP 1: SELECT ACTION using epsilon-greedy policy
            # agent.act(state, eps) implements:
            #   - With probability (1-eps): select argmax Q(state, a) [exploitation]
            #   - With probability eps: select random action [exploration]
            # This balances learning the best strategy while exploring alternative actions
            action = agent.act(state, eps)
            
            # STEP 2: EXECUTE ACTION IN ENVIRONMENT
            # The environment updates internal physics and returns:
            next_state, reward, terminated, truncated, _ = env.step(action)
            
            # STEP 3: COMBINE EPISODE TERMINATION FLAGS
            # Both "terminated" (goal reached/crashed) and "truncated" (max steps) end the episode
            # This boolean will be passed to agent.step() and used in learning
            # In DQN learn(): done=True zeros out Q(next_state) since episode ended
            done = terminated or truncated
            
            # STEP 4: STORE EXPERIENCE AND TRIGGER LEARNING
            # agent.step() performs:
            #   1. Adds (state, action, reward, next_state, done) to replay buffer
            #   2. Every UPDATE_EVERY steps (4 steps): 
            #      - Samples random batch from replay buffer
            #      - Computes Bellman target: r + γ * max_a' Q(s', a')
            #      - Updates local Q-network via gradient descent
            #      - Soft updates target network: θ_target = τ*θ_local + (1-τ)*θ_target
            # This is where learning happens - the network weights are updated!
            agent.step(state, action, reward, next_state, done)
            
            # STEP 5: UPDATE STATE FOR NEXT ITERATION
            # Move to next state for the next timestep in this episode
            # Next state becomes current state
            state = next_state
            
            # STEP 6: ACCUMULATE EPISODE REWARD
            # Add immediate reward to episode return
            # Final score = sum of all rewards in this episode
            score += reward
            
            # STEP 7: CHECK FOR EPISODE TERMINATION
            # If environment indicates episode ended, break out of timestep loop
            # Continue to next episode (this prevents wasting steps)
            if done:
                break
        
        # ======================== END OF EPISODE: UPDATE STATISTICS ========================
        # After episode completes (either done=True or t==max_t), record statistics
        
        # Add episode return to rolling window of last 100 scores
        # This deque automatically maintains only the 100 most recent scores
        scores_window.append(score)
        
        # Add episode return to complete scores list
        # Used for final plotting of all episodes
        scores.append(score)
        
        # ======================== EPSILON DECAY: DECREASE EXPLORATION ========================
        # After each episode, reduce epsilon to encourage more exploitation over time
        # Implementation: eps = max(eps_end, eps_decay * eps)
        #   - Multiplies current epsilon by decay factor (0.995)
        #   - Ensures epsilon never goes below eps_end (0.01 = minimum 1% exploration)
        #   - Decay is exponential: eps_t = eps_start * (eps_decay)^t
        # This schedule shifts from pure exploration early (eps~1.0) to pure exploitation late (eps~0.01)
        eps = max(eps_end, eps_decay*eps)
        
        # ======================== PRINT TRAINING PROGRESS (Every step, overwrite line) ========================
        # Print current episode number and average score over last 100 episodes
        # '\r' = carriage return (overwrite current line instead of new line)
        # end="" prevents automatic newline so progress updates on same line
        print('\rEpisode {}\tAverage Score: {:.2f}'.format(i_episode, np.mean(scores_window)), end="")
        
        # ======================== DETAILED PROGRESS REPORT (Every 100 episodes) ========================
        # Print fuller progress update every 100 episodes for clearer monitoring
        if i_episode % 100 == 0:
            print('\rEpisode {}\tAverage Score: {:.2f}'.format(i_episode, np.mean(scores_window)))
        
        # ======================== CHECK CONVERGENCE: EARLY STOPPING ========================
        # Check if environment is "solved": average score >= 200 over last 100 episodes
        # If yes, save the trained network weights and exit training loop
        # This saves computation time once the agent has learned a good policy
        if np.mean(scores_window) >= 200.0:
            # Print final success message
            # (i_episode - 100) = episode number when average score first reached 200
            # This calculates how many episodes were needed to solve the environment
            print('\nEnvironment solved in {:d} episodes!\tAverage Score: {:.2f}'.format(i_episode - 100, np.mean(scores_window)))
            
            # Save the trained local Q-network weights to file
            # torch.save saves the state_dict (all network parameters) to 'checkpoint.pth'
            # This allows the trained agent to be loaded and reused later
            torch.save(agent.qnetwork_local.state_dict(), 'checkpoint.pth')
            
            # Exit the training loop (stop training once solved)
            break
    
    # ======================== RETURN RESULTS ========================
    # Return list of all episode scores for plotting and analysis
    return scores

# ======================== RUN THE TRAINING ========================
# Execute the DQN training function with default hyperparameters
# This typically takes 30-60 minutes depending on GPU availability
# Training runs for maximum 2000 episodes or until environment is solved
scores = dqn()

# ======================== PLOT TRAINING RESULTS ========================
# Create visualization of agent learning progress
fig = plt.figure()
ax = fig.add_subplot(111)

# Plot all episode scores over training time
# x-axis: episode number
# y-axis: total reward achieved in that episode
# The plot should show increasing trend as agent learns better policy
plt.plot(np.arange(len(scores)), scores)
plt.ylabel('Score')
plt.xlabel('Episode #')
plt.show()

### 4. Watch a Smart Agent!

In the next code cell, you will load the trained weights from file to watch a smart agent!

In [11]:
# load the weights from file
agent.qnetwork_local.load_state_dict(torch.load('checkpoint.pth'))

for i in range(3):
    state, _ = env.reset()
    for j in range(200):
        action = agent.act(state)
        env.render()
        state, reward, terminated, truncated, _ = env.step(action)
        if terminated or truncated:
            break

env.close()

### 5. Explore

In this exercise, you have implemented a DQN agent and demonstrated how to use it to solve an OpenAI Gym environment.  To continue your learning, you are encouraged to complete any (or all!) of the following tasks:
- Amend the various hyperparameters and network architecture to see if you can get your agent to solve the environment faster.  Once you build intuition for the hyperparameters that work well with this environment, try solving a different OpenAI Gym task with discrete actions!
- You may like to implement some improvements such as prioritized experience replay, Double DQN, or Dueling DQN! 
- Write a blog post explaining the intuition behind the DQN algorithm and demonstrating how to use it to solve an RL environment of your choosing.  